In [1]:
import os
os.chdir("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

In [2]:
FEATURE_SETS = {
    "Biometrics_only": ["hr", "hrv", "eda", "wrist_temp"],
    "Environment_only": ["co2_noisy", "lux_noisy", "posture_cm"],
    "Fused_multimodal": ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"],
}

df = pd.read_csv("outputs/unified_features.csv").dropna(
    subset=["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"]
)
le = LabelEncoder()
y = le.fit_transform(df["ground_truth"])
groups = df["subject"].values

In [5]:
from sklearn.model_selection import StratifiedGroupKFold

N_REPEATS = 10
results = {}
for name, feats in FEATURE_SETS.items():
    X = df[feats].values
    fold_f1 = []
    for repeat in range(N_REPEATS):
        sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=repeat)
        for train_idx, test_idx in sgkf.split(X, y, groups=groups):
            scaler = StandardScaler()
            Xtr = scaler.fit_transform(X[train_idx])
            Xte = scaler.transform(X[test_idx])
            clf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
            clf.fit(Xtr, y[train_idx])
            preds = clf.predict(Xte)
            fold_f1.append(f1_score(y[test_idx], preds, average="macro", zero_division=0))
    results[name] = fold_f1
    print(f"{name}: mean macro-F1 = {np.mean(fold_f1):.4f} (SD={np.std(fold_f1):.4f}), n={len(fold_f1)}")

Biometrics_only: mean macro-F1 = 0.7501 (SD=0.0916), n=50
Environment_only: mean macro-F1 = 0.4591 (SD=0.0827), n=50
Fused_multimodal: mean macro-F1 = 0.7306 (SD=0.0912), n=50


In [6]:
from scipy import stats

stat, p = stats.wilcoxon(results["Fused_multimodal"], results["Biometrics_only"])
print(f"\nFused vs Biometrics-only: p={p:.4f}")
if p < 0.05 and np.mean(results["Fused_multimodal"]) > np.mean(results["Biometrics_only"]):
    print("--> Multimodal fusion provides a statistically significant improvement.")
    print("    This IS your headline novelty claim.")
else:
    print("--> No significant fusion benefit found. Report honestly — still a")
    print("    legitimate finding; lean on PBC/generalization as novelty instead.")

pd.DataFrame(results).to_csv("outputs/ablation_results.csv", index=False)


Fused vs Biometrics-only: p=0.0866
--> No significant fusion benefit found. Report honestly — still a
    legitimate finding; lean on PBC/generalization as novelty instead.
